#DAY 7 Databricks Challenge

####Task 1 - Add widgets for parameters

#####Creating Widgets

In [0]:
dbutils.widgets.removeAll()


In [0]:
dbutils.widgets.text(
    "source_path",
    "/Volumes/workspace/default/raw_data/2019-Oct.csv"
)

dbutils.widgets.dropdown(
    "layer",
    "bronze",
    ["bronze", "silver", "gold"]
)


#####Reading Widget Values

In [0]:
source_path = dbutils.widgets.get("source_path")
layer = dbutils.widgets.get("layer")

print("Source Path:", source_path)
print("Layer:", layer)


#**************************************************

####Task 2 - Defining each layers and parameters for each so that we can use widgets to run each layer as required

In [0]:
from pyspark.sql import functions as F

def run_layer(layer_name):
    if layer_name == "bronze":
        print("Running Bronze Layer")

        raw = spark.read.csv(
            source_path,
            header=True,
            inferSchema=True
        )

        raw.withColumn(
            "ingestion_ts", F.current_timestamp()
        ).write.format("delta") \
         .mode("append") \
         .save("/Volumes/workspace/default/challenge/bronze_events")

    elif layer_name == "silver":
        print("Running Silver Layer")

        bronze = spark.read.format("delta") \
            .load("/Volumes/workspace/default/challenge/bronze_events")

        bronze.filter(F.col("price") > 0) \
              .dropDuplicates(["user_session", "event_time"]) \
              .write.format("delta") \
              .mode("overwrite") \
              .save("/Volumes/workspace/default/challenge/silver_events")

    elif layer_name == "gold":
        print("Running Gold Layer")

        silver = spark.read.format("delta") \
            .load("/Volumes/workspace/default/challenge/silver_events")

        silver.groupBy("product_id") \
              .count() \
              .write.format("delta") \
              .mode("overwrite") \
              .save("/Volumes/workspace/default/challenge/gold_products")

    else:
        raise ValueError("Invalid layer")




#####Testing the layer and running Bronze layer

In [0]:
print(layer)

In [0]:
run_layer(layer)

#####Running Silver layer

In [0]:
run_layer("silver")

#####Running Gold layer

#####We have modified the code to include MergeSchema = true which otherwise result in Schema Violation between Silver and Gold

In [0]:
silver = spark.read.format("delta") \
    .load("/Volumes/workspace/default/challenge/silver_events")

silver.groupBy("product_id") \
      .count() \
      .write.format("delta") \
      .option("mergeSchema", "true") \
      .mode("overwrite") \
      .save("/Volumes/workspace/default/challenge/gold_products")

In [0]:
run_layer("gold")

#**************************************************


%md
## **For me more such learning and insights in**
- ### [LinkedIn](https://www.linkedin.com/in/ilakkiyan-av/) 
- ### [Youtube](https://www.youtube.com/@ilakkiyanav) 